<hr style="border:1px solid #FEC60D">
<center><h1 style="color:#173F8A;">
Magíster en Inteligencia Artificial (MIA) <br>
Instituto de Ingeniería Matemática y Computacional (IMC)<br>
Pontificia Universidad Católica de Chile
</h1></center>
<hr style="border:1px solid #FEC60D">

<h3 style="color:#03122E;text-align:left;">INF3862-1 - Plataformas Big Data para Data Science</h3>

<center><h3 style="color:#0176DE;">
Proyecto del curso, Entregas 2 y 3<br>Hábitos de actividad y alimentación asociados a mejores indicadores de salud
</h3></center>

<center><h4 style="color:#707070;">
<b>Integrantes:</b><br>
Miguel Angel Cervantes Buleje<br>
Roger Ruben Reyes Miranda<br>
</h4></center>

<center><span style="color:#707070;">Noviembre 2026</span></center>

## Problema, entorno y datos

El dashboard del equipo de nutrición compara, por cuartil de frecuencia cardíaca en reposo, los pasos, los minutos de sesión y los nutrientes de las recetas que preparan los usuarios (E1, sección 4.1). Este notebook lleva ese diseño a un pipeline con DuckDB sobre los datos completos del curso: 61 días de mediciones de los 300 usuarios con reloj. El código está en `src/pipeline.py` (tareas del DAG), `src/consultas.py` (consultas 4, 5 y 6 de la E1) y `src/modelamiento.py` (E3).

Los datos se obtienen según `datos/README.md`: descargar `datos_proyecto_INF3862.zip` desde el curso y descomprimirlo en `datos/`. La capa bronze es esa carpeta, sin cambios; el pipeline escribe las capas silver y gold en `datos/silver/` y `datos/gold/`.

In [1]:
import platform
import sys
from pathlib import Path

import duckdb
import pandas as pd

# raíz del repositorio: la carpeta que tiene pyproject.toml
RAIZ = next(c for c in (Path.cwd(), *Path.cwd().parents) if (c / "pyproject.toml").exists())
# los módulos de la carpeta del notebook si están (copia fija de una entrega); si no, los de src/
CODIGO = Path.cwd() if (Path.cwd() / "pipeline.py").exists() else RAIZ / "src"
sys.path.insert(0, str(CODIGO))

import modelamiento
from consultas import CONSULTAS
from pipeline import USUARIOS_FOODCOM, Pipeline, cpus_visibles, medir, megabytes

meminfo = Path("/proc/meminfo")
memoria_gb = int(meminfo.read_text().split()[1]) / 1e6 if meminfo.exists() else float("nan")
print("Python", platform.python_version(), "| DuckDB", duckdb.__version__, "| pandas", pd.__version__)
print("CPU visibles:", cpus_visibles(), f"| memoria: {memoria_gb:.1f} GB | sistema:", platform.system())
print("Código desde:", CODIGO.relative_to(RAIZ))

Python 3.12.12 | DuckDB 2.0.0 | pandas 3.1.1
CPU visibles: 12 | memoria: 10.2 GB | sistema: Linux
Código desde: src


## 5.2 Implementación mínima

Cada paso es una tarea del DAG de la sección 5.1 del informe y se ejecuta con `p.tarea`, que guarda su tiempo.

### Paso 1. Cargar los datos crudos tal como vienen

In [2]:
p = Pipeline(RAIZ)
p.tarea("cargar_bronze", p.cargar_bronze)
bronze = pd.DataFrame(p.resumen_bronze(), columns=["fuente", "formato", "archivos", "filas", "MB"]).round(1)
bronze

,fuente,formato,archivos,filas,MB
0,mediciones,JSONL.GZ,1449,15540132,115.5
1,dispositivos,CSV,1,315,0.0
2,sesiones,JSONL,1,4840,0.8
3,comidas,CSV,1,22106,1.0
4,recetas,Parquet,1,522517,178.7


**¿Cómo se cargan los datos crudos tal como vienen?** DuckDB lee desde `datos/datos_proyecto_INF3862/` los 1449 archivos de mediciones en JSON comprimido, los dos CSV, el JSONL de sesiones y el Parquet de recetas, sin modificarlos ni moverlos. Los dos modelos de reloj nombran distinto los mismos campos (`device_id` y `dev`; `ts_medicion` en ISO 8601 y `timestamp` en milisegundos): la lectura los unifica en un solo esquema y no descarta filas, de modo que las 15540132 mediciones llegan completas a las reglas del paso 2.

### Paso 2. Reglas de calidad

In [3]:
p.tarea("construir_silver", p.construir_silver)
calidad = pd.DataFrame(p.calidad.items(), columns=["regla o conteo", "registros"])
print("Atraso máximo entre medición y recepción:", p.atraso_maximo_dias, "días")
calidad

Atraso máximo entre medición y recepción: 4 días


,regla o conteo,registros
0,mediciones leídas,15540132
1,tipos: sin marca de tiempo o reloj,0
2,claves: reloj fuera de dispositivos.csv,0
3,duplicados: mismo reloj y marca de tiempo,345599
4,duplicados con valores distintos,0
5,claves: medición fuera del vínculo del reloj,0
6,nulos: frecuencia cardíaca nula,349931
7,rangos: frecuencia cardíaca fuera de 20-250,0
8,rangos: pasos o calorías negativos,0
9,atrasos: recibidas más de 1 hora después,8957451


**¿Qué reglas de calidad se aplican y cuántos registros afecta cada una?** Seis reglas, que en estos datos afectan a duplicados, nulos y claves; tipos y rangos no tienen casos:

- **Tipos:** toda medición tiene reloj y marca de tiempo después de unificar los esquemas (0 registros).
- **Claves que no calzan:** 0 mediciones de relojes que no están en `dispositivos.csv` y 0 fuera del período en que su reloj estaba vinculado. 48 comidas tienen una receta que no está en el catálogo: salen de silver, porque sin receta no tienen nutrientes.
- **Duplicados:** 345599 mediciones repiten reloj y marca de tiempo, el 2.2% de las 15540132. Ninguna repetida tiene valores distintos (0): son lotes reenviados y se conserva la primera recepción. 48 sesiones repiten `session_id`; sin esta regla sus minutos se sumarían dos veces. Ninguna comida se repite.
- **Nulos:** 349931 mediciones no traen frecuencia cardíaca, el 2.3% de las 15194533 sin repetidos. Se conservan, porque sus pasos y calorías valen, pero no entran al promedio de reposo. La E1 cuenta 357927 porque los cuenta antes de quitar los repetidos.
- **Rangos válidos:** 0 frecuencias fuera de 20-250 latidos por minuto y 0 pasos o calorías negativos.
- **Atrasos:** 8957451 mediciones llegan más de una hora después de medirse y 2185094 llegan otro día UTC, con hasta 4 días de atraso. No se descartan: se guardan en el día en que se midieron (paso 3).

### Paso 3. Capa limpia en Parquet, particionada según la E1

In [4]:
particiones = sorted(d.name for d in (p.silver / "mediciones").iterdir())
print("Particiones de mediciones:", len(particiones), "| primera:", particiones[0], "| última:", particiones[-1])
print("Mediciones en silver:", p.valor(f"select count(*) from {p.tabla(p.silver, 'mediciones')}")[0],
      f"| MB: {megabytes(p.silver / 'mediciones'):.1f} | MB en bronze: {megabytes(p.bronze / 'wearable' / 'mediciones'):.1f}")
print("Sesiones y comidas en silver:", p.valor(f"select count(*) from {p.tabla(p.silver, 'sesiones')}")[0], "|",
      p.valor(f"select count(*) from {p.tabla(p.silver, 'comidas')}")[0])

Particiones de mediciones: 61 | primera: fecha_medicion=2007-12-16 | última: fecha_medicion=2008-02-14


Mediciones en silver: 15194533 | MB: 52.1 | MB en bronze: 115.5
Sesiones y comidas en silver: 4792 | 22058


**¿Cómo se guarda la capa limpia?** Silver guarda las 15194533 mediciones sin repetidos en Parquet con compresión ZSTD, particionadas por fecha de medición como propuso la E1: 61 carpetas, del 2007-12-16 al 2008-02-14. Ocupan 52.1 MB, menos de la mitad de los 115.5 MB de JSON comprimido de bronze. Cada medición atrasada queda en la carpeta del día en que se midió, de modo que recalcular un día lee solo esa carpeta. Las 4792 sesiones y las 22058 comidas van sin partición, porque ocupan menos de 1 MB cada una.

### Paso 4. Tabla de hechos y dimensiones de la E1

In [5]:
p.tarea("construir_gold", p.construir_gold)
p.registrar_vistas_gold()
TABLAS_GOLD = ["hecho_usuario_dia", "hecho_usuario_reloj_dia", "hecho_comida",
               "dim_fecha", "dim_usuario", "dim_reloj", "dim_receta"]
gold = pd.DataFrame([(t, p.valor(f"select count(*) from {t}")[0],
                      len([d for d in (p.gold / t).iterdir() if d.is_dir()]))
                     for t in TABLAS_GOLD], columns=["tabla", "filas", "particiones por mes"])
# días sin mediciones: antes de vincular el primer reloj o con un reloj vinculado que no envió datos
print(p.sql("""
    with vinculo as (select usuario_key, min(cast(desde as date)) as desde from dim_reloj group by 1)
    select count(*) filter (where h.pasos is null) as dias_sin_mediciones,
           count(*) filter (where h.pasos is null and h.fecha_key < v.desde) as antes_de_vincular,
           count(*) filter (where h.fc_reposo is not null) as dias_con_fc_reposo,
           count(distinct h.usuario_key) filter (where h.fc_reposo is not null) as usuarios_con_fc_reposo
    from hecho_usuario_dia h join vinculo v using (usuario_key)""").df().to_string(index=False))
gold

 dias_sin_mediciones  antes_de_vincular  dias_con_fc_reposo  usuarios_con_fc_reposo
                1167                674               11823                     216


,tabla,filas,particiones por mes
0,hecho_usuario_dia,17497,3
1,hecho_usuario_reloj_dia,16330,3
2,hecho_comida,22058,3
3,dim_fecha,61,0
4,dim_usuario,300,0
5,dim_reloj,315,0
6,dim_receta,20654,0


**¿Cómo se construyen la tabla de hechos y las dimensiones de la E1?** Gold guarda en Parquet los tres hechos y las cuatro dimensiones del anexo B de la E1, con sus nombres y claves (`usuario_key`, `fecha_key`, `reloj_key`, `receta_key`). Los hechos se particionan por mes, como en la E1 (3 carpetas: diciembre, enero y febrero); las dimensiones van sin partición.

- **`hecho_usuario_dia`:** 17497 filas, una por usuario y día local con mediciones o con comidas. En 1167 días hay comidas pero no mediciones: 674 son anteriores a la vinculación del primer reloj y el resto son días en que el reloj vinculado no envió datos. En esos días los pasos quedan nulos y no en cero, porque el reloj no midió.
- **Frecuencia cardíaca en reposo:** la tienen 11823 días de 216 usuarios, el mismo universo de la E1. La E1 cuenta 11864 días con datos de esos usuarios, de los cuales 41 no tienen mediciones de madrugada y quedan nulos: 11864 - 41 = 11823.
- **`hecho_usuario_reloj_dia`:** 16330 filas, una por usuario, día y reloj, para no mezclar los pasos de dos modelos en los días en que un usuario cambia de reloj.
- **`hecho_comida`:** 22058 comidas con receta en el catálogo y sus nutrientes consumidos (nutrientes por porción por las porciones de la comida).
- **Dimensiones:** 61 fechas, 300 usuarios, 315 relojes y las 20654 recetas que aparecen en las comidas.

### Paso 5. Consultas analíticas de la E1

In [6]:
resultados = {nombre: p.sql(consulta).df() for nombre, consulta in CONSULTAS.items()}
resultados["consulta_4"]

,cuartil,usuarios,fc_reposo,pasos,minutos_sesion,sodio,grasa_saturada,fibra
0,1,54,51.4,8515.0,14.6,4251.0,49.6,17.4
1,2,54,58.4,8070.0,13.4,4040.0,45.0,17.4
2,3,54,63.0,8253.0,14.6,3890.0,47.3,15.9
3,4,54,69.3,8822.0,16.0,4588.0,53.6,18.1


**¿Qué hábitos tiene cada cuartil de frecuencia cardíaca en reposo?** Los 216 usuarios quedan en cuartiles de 54, con promedios de 51.4 a 69.3 latidos por minuto, y los hábitos no siguen el orden de los cuartiles:

- **Actividad:** el cuartil 1 (frecuencia más baja) camina 8515 pasos diarios y el cuartil 4, 8822; el que menos camina es el cuartil 2 (8070). Los minutos de sesión van de 13.4 a 16.0.
- **Alimentación:** el cuartil 4 consume más sodio (4588 mg), más grasa saturada (53.6 g) y también más fibra (18.1 g) que los otros tres.
- **Interpretación:** la actividad no ordena a los cuartiles, y el cuartil 4 consume más de los tres nutrientes, incluida la fibra. Es una descripción de datos sintéticos, no una relación causal.

In [7]:
resultados["consulta_5"].pivot(index="semana", columns="modelo", values="pasos_promedio")

modelo,FitBand 2,Pulso S
semana,,
2007-W50,8367.0,8055.0
2007-W51,8445.0,8503.0
2007-W52,8464.0,8232.0
2008-W01,8424.0,8246.0
2008-W02,8420.0,8258.0
2008-W03,8390.0,8367.0
2008-W04,8698.0,8378.0
2008-W05,8315.0,8588.0
2008-W06,8277.0,8422.0


**¿Reportan pasos comparables los dos modelos de reloj?** Sí. Cada semana, la diferencia entre los promedios de los dos modelos va de 23 pasos (2008-W03) a 458 (2008-W07), y FitBand 2 supera a Pulso S en 7 de las 10 semanas. La semana con la mayor diferencia, 2008-W07, y la primera, 2007-W50, están incompletas: tienen 314 y 80 filas de FitBand 2, frente a 595-751 en las semanas completas. Por lo tanto, la consulta 4 puede juntar usuarios de los dos modelos sin corregir sus pasos.

In [8]:
# comidas que cuenta la consulta 6: las de usuarios con cuartil
comidas_c6 = p.valor("""select count(*) from hecho_comida where usuario_key in
                       (select usuario_key from hecho_usuario_dia where fc_reposo is not null)""")[0]
print("Comidas de usuarios con cuartil:", comidas_c6, "| máximo de veces de una receta en un cuartil:",
      resultados["consulta_6"]["veces"].max())
# nombres compartidos por más de una receta: la consulta 6 agrupa también por receta_key
nombres_repetidos = p.valor("select count(*) from (select nombre from dim_receta group by 1 having count(*) > 1)")[0]
print("Nombres de receta que usan dos o más recetas:", nombres_repetidos)
resultados["consulta_6"].groupby("cuartil").head(3)

Comidas de usuarios con cuartil: 15889 | máximo de veces de una receta en un cuartil: 4
Nombres de receta que usan dos o más recetas: 469


,cuartil,nombre,veces
0,1,Bourbon Chicken,4
1,1,Spinach Artichoke Dip,3
2,1,Bacon Potato Frittata,2
10,2,Chocolate Cream Cheese Pound Cake,3
11,2,5 Can Ww Soup,2
12,2,Absolute Best Ever Lasagna,2
20,3,Authentic Italian Meatballs,3
21,3,Simple Whole Wheat Muffins,3
22,3,2bleu's Sweet Chicken and Gravy,2
30,4,Applesauce Oatmeal,3


**¿Qué recetas prepara más cada cuartil?** La receta que más se repite es Bourbon Chicken, 4 veces en el cuartil 1, entre 15889 comidas de usuarios con cuartil (el mismo conteo de la E1). La consulta agrupa también por `receta_key`, porque 469 nombres corresponden a dos o más recetas distintas. Con tan pocas repeticiones, el top 10 de cada cuartil termina en empates de 2 veces, que la consulta ordena por nombre. Por lo tanto, el ranking de recetas no distingue a los cuartiles: el dashboard los compara mejor por nutrientes (consulta 4).

### Reintentos e idempotencia (5.1)

In [9]:
# repetir las tareas silver y gold debe dejar las mismas filas, sin duplicar
antes = {t: p.valor(f"select count(*) from {t}")[0] for t in TABLAS_GOLD}
p.construir_silver()
p.construir_gold()
p.registrar_vistas_gold()
despues = {t: p.valor(f"select count(*) from {t}")[0] for t in TABLAS_GOLD}
print("Mismas filas después de repetir:", antes == despues, "|", despues["hecho_usuario_dia"], "filas en hecho_usuario_dia")

Mismas filas después de repetir: True | 17497 filas en hecho_usuario_dia


**¿Cómo se evita duplicar registros si una carga se repite?** Volver a ejecutar silver y gold deja las mismas filas en las siete tablas (17497 en `hecho_usuario_dia`): cada tarea borra y vuelve a escribir completa la carpeta que produce, y silver quita los repetidos por reloj y marca de tiempo antes de escribir. Por lo tanto, un reintento después de una falla deja el mismo estado final.

## 5.3 Medir y explicar

In [10]:
# consulta 4 sobre hecho_usuario_dia guardado en CSV y en Parquet
csv_hecho = p.gold / "csv" / "hecho_usuario_dia.csv"
csv_hecho.parent.mkdir(exist_ok=True)
p.sql(f"copy (select * from hecho_usuario_dia) to '{csv_hecho}' (header)")
consulta_csv = CONSULTAS["consulta_4"].replace("FROM hecho_usuario_dia", f"FROM read_csv_auto('{csv_hecho}')")
consulta_parquet = CONSULTAS["consulta_4"]
mismo_resultado = p.sql(consulta_csv).df().equals(p.sql(consulta_parquet).df())

formatos = pd.DataFrame({
    "formato": ["CSV", "Parquet"],
    "MB en disco": [megabytes(csv_hecho.parent), megabytes(p.gold / "hecho_usuario_dia")],
    "ms (mediana de 7)": [medir(lambda: p.sql(consulta_csv).fetchall()),
                          medir(lambda: p.sql(consulta_parquet).fetchall())],
}).round(2)
print("Mismo resultado en CSV y Parquet:", mismo_resultado)
formatos

Mismo resultado en CSV y Parquet: True


,formato,MB en disco,ms (mediana de 7)
0,CSV,2.15,254.43
1,Parquet,0.52,32.17


In [11]:
# las columnas que lee la consulta 4 frente a las que tiene el hecho
columnas_hecho = p.sql("select * from hecho_usuario_dia limit 0").columns
columnas_consulta = ["usuario_key", "fecha_key", "fc_reposo", "pasos", "minutos_sesion", "sodio", "grasa_saturada", "fibra"]
print("Columnas del hecho:", len(columnas_hecho), "| que lee la consulta 4:", len(columnas_consulta))
print("Filas del hecho:", p.valor("select count(*) from hecho_usuario_dia")[0],
      "| archivos Parquet:", len(list((p.gold / "hecho_usuario_dia").rglob("*.parquet"))))

Columnas del hecho: 18 | que lee la consulta 4: 8
Filas del hecho: 17497 | archivos Parquet: 3


**¿Cuánto tarda la consulta 4 sobre CSV y sobre Parquet, y por qué?** Sobre Parquet tarda 32.17 ms, 7.9 veces menos que sobre CSV (254.43 ms), y ocupa 0.52 MB, 4.1 veces menos que el CSV (2.15 MB). Las dos dan el mismo resultado. Los tiempos son la mediana de 7 ejecuciones en el contenedor de la sección de entorno.

- **Organización por columnas:** la consulta usa 8 de las 18 columnas del hecho. Parquet guarda cada columna por separado y DuckDB lee solo esas 8; el CSV guarda filas de texto, y para cada fila hay que leer las 18 columnas y convertir el texto en números y fechas.
- **Compresión:** Parquet guarda juntos los valores de una columna, que se parecen entre sí (fechas seguidas, nulos de los días sin comidas), y los comprime con ZSTD. Por eso ocupa 4.1 veces menos.
- **Poda de particiones:** no participa en esta medición. La consulta pide el período completo y lee los 3 meses; con un filtro de un mes leería un solo archivo de los 3.

## 5.4 ¿Realmente necesitamos Big Data?

In [12]:
DIAS_ANIO = 365
usuarios_reloj = p.valor("select count(distinct AuthorId) from dispositivos")[0]
usuarios_foodcom = p.valor(f"select count(distinct AuthorId) from read_parquet('{p.bronze / USUARIOS_FOODCOM}')")[0]
dias = p.dias_recepcion
filas_crudas = p.calidad["mediciones leídas"]
filas_usuario_dia = filas_crudas / usuarios_reloj / dias
mb_bronze_fila = megabytes(p.bronze / "wearable" / "mediciones") / filas_crudas
mb_silver_fila = megabytes(p.silver / "mediciones") / p.calidad["mediciones en silver"]
segundos_fila = (p.tiempos["cargar_bronze"] + p.tiempos["construir_silver"]) / filas_crudas
print(f"Usuarios con reloj: {usuarios_reloj} | usuarios de Food.com: {usuarios_foodcom} | días: {dias}")
print(f"Mediciones por usuario y día: {filas_usuario_dia:.1f} | bytes por medición: bronze {mb_bronze_fila * 1e6:.2f}, silver {mb_silver_fila * 1e6:.2f}")
print(f"Tiempo de bronze y silver: {p.tiempos['cargar_bronze'] + p.tiempos['construir_silver']:.1f} s "
      f"| memoria máxima del proceso: {p.memoria_maxima_gb():.1f} GB de {memoria_gb:.1f} GB")

escenarios = {"hoy (300 usuarios, 61 días)": (usuarios_reloj, dias),
              "Food.com, 1 año": (usuarios_foodcom, DIAS_ANIO),
              "Food.com, 3 años": (usuarios_foodcom, 3 * DIAS_ANIO),
              "Food.com x100, 3 años": (100 * usuarios_foodcom, 3 * DIAS_ANIO)}
escala = pd.DataFrame([{
    "escenario": nombre,
    "mediciones por día (millones)": usuarios * filas_usuario_dia / 1e6,
    "mediciones (miles de millones)": usuarios * filas_usuario_dia * d / 1e9,
    "bronze (GB)": usuarios * filas_usuario_dia * d * mb_bronze_fila / 1e3,
    "silver (GB)": usuarios * filas_usuario_dia * d * mb_silver_fila / 1e3,
    "lote diario (min)": usuarios * filas_usuario_dia * segundos_fila / 60,
    "reproceso completo (h)": usuarios * filas_usuario_dia * d * segundos_fila / 3600,
} for nombre, (usuarios, d) in escenarios.items()])
escala.round(2)

Usuarios con reloj: 300 | usuarios de Food.com: 271907 | días: 61
Mediciones por usuario y día: 849.2 | bytes por medición: bronze 7.43, silver 3.43
Tiempo de bronze y silver: 20.8 s | memoria máxima del proceso: 6.0 GB de 10.2 GB


,escenario,mediciones por día (millones),mediciones (miles de millones),bronze (GB),silver (GB),lote diario (min),reproceso completo (h)
0,"hoy (300 usuarios, 61 días)",0.25,0.02,0.12,0.05,0.01,0.01
1,"Food.com, 1 año",230.90,84.28,626.39,289.33,5.15,31.33
2,"Food.com, 3 años",230.90,252.84,1879.17,868.00,5.15,94.00
3,"Food.com x100, 3 años",23090.00,25283.55,187916.50,86799.82,515.09,9400.35


**¿Qué volumen tendría la app con todos los usuarios de Food.com, y cuánto tardaría en una máquina?** Supuesto: los 271907 usuarios que reseñan en Food.com tienen reloj y generan, como los 300 actuales, 849.2 mediciones por día. Los tamaños y tiempos se extrapolan en forma lineal desde esta corrida, que tardó 20.8 s en bronze y silver (bytes por medición y segundos de bronze y silver por medición); son estimaciones, no mediciones.

- **1 año:** 84.28 mil millones de mediciones, 626.39 GB en bronze y 289.33 GB en silver. El lote diario (230.9 millones de mediciones) tardaría unos 5.15 minutos.
- **3 años:** 252.84 mil millones de mediciones, 1879.17 GB en bronze y 868.00 GB en silver. El lote diario es el mismo; volver a procesar los 3 años tardaría unas 94.0 horas.
- **x100 a 3 años:** el lote diario tardaría 515.09 minutos (8.6 horas) y bronze ocuparía 187916.5 GB.
- **Memoria:** esta corrida ocupó 6.0 GB de los 10.2 GB del contenedor con 15.5 millones de mediciones, porque bronze y silver las guardan en tablas en memoria. Un lote diario de todos los usuarios tiene 15 veces más mediciones que los 61 días actuales.

La respuesta sobre el procesamiento distribuido está en la sección 5.4 del informe.

## 6.2 Tabla de modelamiento y modelo base (E3)

`src/modelamiento.py` arma, desde `hecho_usuario_dia` de la capa gold, una fila por usuario y día con las variables de los 7 días anteriores y la frecuencia cardíaca en reposo del día siguiente como objetivo. Compara una regresión lineal con la mediana del período de entrenamiento.

In [13]:
metricas = modelamiento.run_model(RAIZ)
pd.DataFrame({"mediana de entrenamiento": metricas["baseline"], "regresión lineal": metricas["ols"]})

{
  "objetivo": "frecuencia cardíaca en reposo del día siguiente (bpm)",
  "ventana_features": "7 días anteriores; no incluye frecuencia cardíaca como predictor",
  "particion": "temporal global: train hasta 2008-02-02; test posterior. Los usuarios pueden aparecer en ambos periodos; ninguna variable posterior a fecha_features_hasta se usa como predictor",
  "fecha_corte_temporal": "2008-02-02",
  "features": [
    "pasos_7d_promedio",
    "minutos_sesion_7d_promedio",
    "comidas_7d",
    "porciones_7d",
    "calorias_7d",
    "sodio_7d",
    "grasa_saturada_7d",
    "azucar_7d",
    "fibra_7d",
    "proteina_7d"
  ],
  "nutrition_feature_policy": "7-day sums over matched, logged meals; no-meal days contribute zero recorded intake; windows with meal events missing nutrient joins are excluded",
  "n_rows_total": 7871,
  "n_rows_train": 6226,
  "n_rows_test": 1645,
  "n_users_train": 216,
  "n_users_test": 213,
  "usuarios_compartidos_train_test": 213,
  "baseline_mediana_train": 60.937

,mediana de entrenamiento,regresión lineal
mae_bpm,5.6166,5.6378
rmse_bpm,7.1423,7.1960
r2,-0.0005,-0.0156


**¿Supera el modelo base a la referencia trivial?** No. Sobre las fechas posteriores al 2008-02-02, la regresión lineal tiene un error absoluto medio de 5.6378 latidos por minuto, frente a 5.6166 de predecir siempre la mediana de entrenamiento, y su $R^2$ es negativo (-0.0156): predice peor que el promedio del período de evaluación. La E3 discute este resultado.